# XGBoost + NN Predictor Features — Binary Confidence Analysis

Binary short/long XGBoost training with Chronos/Kronos/TimesFM predictor features, calibration diagnostics, confidence-band checks, and threshold sweeps.


In [ ]:
import os
import json
import tempfile
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import xgboost as xgb
import mlflow
import mlflow.xgboost

from pathlib import Path
from sklearn.metrics import (
    roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score,
    f1_score, precision_score, recall_score,
    confusion_matrix, classification_report,
    log_loss, brier_score_loss, balanced_accuracy_score,
    matthews_corrcoef,
)
from sklearn.calibration import calibration_curve

from Pipeline.pipeline import ForexDataLoader, ForexPipeline

sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

## Configuration

Edit `DATA_CFG`, `NN_FEATURES_CFG`, and `XGB_PARAMS` here.

**`DATA_CFG`** — pipeline parameters; also controls the shared filename tags for selected NN feature parquets
(`norm_method`, `weekends`, `fracdiff_d`, `scaling`, and `years`).

**`NN_FEATURES_CFG`** — provider-specific switches and inference parameters for Chronos, Kronos,
and TimesFM. Enable any one provider, any pair, or all three. Do **not** duplicate shared
normalization/weekend/scaling/year settings here; those are read from `DATA_CFG`.


In [ ]:
DATA_CFG = {
    "pair":             "EURUSD",
    "years":            [2023],
    "timeframe":        "H1",
    "weekends":         "filled",   # "filled" | "nogap" | "gaps"
    # normalization: "log_returns" | "fracdiff" | "raw"
    "norm_method":      "fracdiff",
    # target: "lag" (direction_1/5/15) | "triple_barrier" (tb_label)
    "target_type":      "triple_barrier",
    # target column: "tb_label" for triple_barrier | "direction_1/5/15" for lag
    "target_col":       "tb_label",
    "lags":             [1, 2, 5, 10],
    "target_horizons":  [1, 5, 15],  # pipeline target horizons used as features
    "gap_bars":         50,
    # scaling: "global" (ForexScaler) | "rolling" (RollingScaler) | "none"
    "scaling":          "none",
    "window_size":      500,         # only used if scaling="rolling"
    "fracdiff_d":       0.3,
    "threshold":        6 * 1e-4,
    "k_up":             1.0,
    "k_down":           1.0,
    "horizon_bars":     10,          # target horizon to predict
    "barrier_price":    "hl",
    "barrier_norm_method": "raw",
}

# NN feature params. Keep norm_method / weekends / scaling / years in DATA_CFG.
# Enable any one provider, any pair of providers, or all three.
NN_FEATURES_CFG = {
    "chronos": {
        "enabled":        True,
        "context_length": 504,
        "calc_interval":  5,
        "horizons":       [5, 10, 15, 20],
        "percentiles":    [0.05, 0.25, 0.50, 0.75, 0.95],
        "use_staleness":  False,
    },
    "kronos": {
        "enabled":        True,
        "context_length": 512,
        "calc_interval":  5,
        "horizons":       [5, 10, 15, 20],
        # n_samples == 1: OHLC/spread columns; n_samples > 1: quantile columns.
        "n_samples":      1,
        "percentiles":    [0.05, 0.25, 0.50, 0.75, 0.95],
        "temperature":    1.0,
        "top_p":          0.9,
        "use_staleness":  False,
    },
    "timesfm": {
        "enabled":        True,
        "context_length": 512,
        "calc_interval":  5,
        "horizons":       [5, 10, 15, 20],
        # TimesFM supports fixed levels: 0.0 mean, then 0.1 ... 0.9 quantiles.
        "percentiles":    [0.1, 0.3, 0.5, 0.7, 0.9],
        "use_staleness":  False,
    },
}

XGB_PARAMS = {
    "n_estimators":          1000,
    "max_depth":             3,
    "learning_rate":         0.03,
    "subsample":             0.8,
    "colsample_bytree":      0.8,
    "min_child_weight":      20,
    "gamma":                 0.0,
    "reg_alpha":             0.0,
    "reg_lambda":            2.0,
    "early_stopping_rounds": 30,
    "objective":             "binary:logistic",
    "eval_metric":           "logloss",
    "random_state":          42,
    "n_jobs":                -1,
}

CONFIDENCE_CFG = {
    "splits": ["val", "test"],
    "bands": [
        {"name": "long_55_70", "side": "long", "min_p": 0.55, "max_p": 0.70},
        {"name": "short_55_70", "side": "short", "min_p": 0.55, "max_p": 0.70},
    ],
}

DIAGNOSTIC_CFG = {
    "calibration_bins": 10,
    "thresholds": [0.50, 0.52, 0.55, 0.57, 0.60, 0.62, 0.65, 0.70, 0.75, 0.80],
    "threshold_splits": ["val", "test"],
    "min_coverage_warn": 0.01,
    "near_random_accuracy_warn": 0.53,
}

MLFLOW_DB       = f"sqlite:///{os.path.abspath('mlflow.db')}"
EXPERIMENT_NAME = "xgboost_nns_forex_binary_conf"

enabled_nn_sources = [name for name, cfg in NN_FEATURES_CFG.items() if cfg.get("enabled", False)]
assert enabled_nn_sources, "Enable at least one provider in NN_FEATURES_CFG."

print(f"DATA_CFG       : {json.dumps(DATA_CFG, indent=2)}")
print(f"NN_FEATURES_CFG: {json.dumps(NN_FEATURES_CFG, indent=2)}")
print(f"Enabled NN     : {enabled_nn_sources}")
print(f"XGB_PARAMS     : {json.dumps(XGB_PARAMS, indent=2)}")
print(f"MLflow DB      : {MLFLOW_DB}")
print(f"CONFIDENCE_CFG : {json.dumps(CONFIDENCE_CFG, indent=2)}")
print(f"DIAGNOSTIC_CFG : {json.dumps(DIAGNOSTIC_CFG, indent=2)}")


## 0 — Resolve NN Feature Parquet Paths


In [ ]:
def _norm_tag(data_cfg: dict) -> str:
    nm = data_cfg["norm_method"]
    if nm == "log_returns":
        return "logret"
    if nm == "fracdiff":
        return f"fdiff{data_cfg['fracdiff_d']}"
    return "raw"


def _h_tag(cfg: dict) -> str:
    return "h" + "-".join(str(h) for h in sorted(set(cfg["horizons"])))


def _scale_tag(data_cfg: dict) -> str:
    scaling = data_cfg["scaling"]
    if scaling == "none":
        return "snone"
    if scaling == "rolling":
        return f"sroll{data_cfg['window_size']}"
    return "sglob"


def _year_tag(data_cfg: dict) -> str:
    years = data_cfg.get("years")
    if years:
        return "_".join(str(y) for y in sorted(years))
    start = data_cfg.get("start")
    end = data_cfg.get("end")
    if start or end:
        return f"{(start or '').replace('-', '')}-{(end or '').replace('-', '')}"
    return "all"


def _build_nn_fname(source: str, data_cfg: dict, cfg: dict) -> str:
    pair = data_cfg["pair"]
    tf = data_cfg["timeframe"]
    ctx = cfg["context_length"]
    interval = cfg["calc_interval"]
    base = f"{pair}_{tf}"
    suffix = (
        f"_ctx{ctx}_int{interval}_{_h_tag(cfg)}"
        f"_{_norm_tag(data_cfg)}_w{data_cfg['weekends']}"
        f"_{_scale_tag(data_cfg)}_{_year_tag(data_cfg)}.parquet"
    )

    if source == "chronos":
        return f"{base}{suffix}"
    if source == "timesfm":
        return f"{base}_tfm{suffix}"
    if source == "kronos":
        sample_tag = f"_s{cfg['n_samples']}" if cfg.get("n_samples", 1) > 1 else ""
        return (
            f"{base}_kron_ctx{ctx}_int{interval}_{_h_tag(cfg)}"
            f"_{_norm_tag(data_cfg)}_w{data_cfg['weekends']}"
            f"_{_scale_tag(data_cfg)}{sample_tag}_{_year_tag(data_cfg)}.parquet"
        )
    raise ValueError(f"Unknown NN feature source: {source}")


def _generate_hint(source: str, data_cfg: dict, cfg: dict) -> str:
    imports = {
        "chronos": "from Chronos.chronos_features import generate",
        "kronos": "from Kronos.kronos_features import generate",
        "timesfm": "from Timeseriesfm.timesfm_features import generate",
    }
    args = {
        "pair": data_cfg["pair"],
        "years": data_cfg["years"],
        "timeframe": data_cfg["timeframe"],
        "context_length": cfg["context_length"],
        "calc_interval": cfg["calc_interval"],
        "horizons": cfg["horizons"],
        "norm_method": data_cfg["norm_method"],
        "fracdiff_d": data_cfg["fracdiff_d"],
        "threshold": data_cfg["threshold"],
        "weekends": data_cfg["weekends"],
        "scaling": data_cfg["scaling"],
        "scaling_window": data_cfg["window_size"],
    }
    if cfg.get("percentiles") is not None:
        args["percentiles"] = cfg["percentiles"]
    if source == "kronos":
        args["n_samples"] = cfg["n_samples"]
        args["temperature"] = cfg["temperature"]
        args["top_p"] = cfg["top_p"]
    rendered = ",\n    ".join(f"{k}={v!r}" for k, v in args.items())
    return f"{imports[source]}\ngenerate(\n    {rendered},\n)"


nn_feature_paths = {}
for source, cfg in NN_FEATURES_CFG.items():
    if not cfg.get("enabled", False):
        continue
    fname = _build_nn_fname(source, DATA_CFG, cfg)
    path = Path("featdata") / fname
    nn_feature_paths[source] = {"fname": fname, "path": path}
    print(f"{source:8s} expected parquet: {path}")
    if not path.exists():
        raise FileNotFoundError(
            f"\n\n{source} parquet not found: {path}\n"
            f"This notebook is load-only. Generate it first with:\n"
            f"{_generate_hint(source, DATA_CFG, cfg)}"
        )
    _probe = pd.read_parquet(path)
    print(f"  rows    : {len(_probe):,}")
    print(f"  columns : {list(_probe.columns)}")
    print(f"  range   : {_probe.index.min()} -> {_probe.index.max()}")
    del _probe


## 1 — Load Raw M1 Data

In [ ]:
loader = ForexDataLoader()
df_m1  = loader.load_and_merge(
    "histdata/",
    pair=DATA_CFG["pair"],
    years=DATA_CFG["years"],
    weekends=DATA_CFG["weekends"],
)

print(f"Raw M1 shape : {df_m1.shape}")
print(f"Date range   : {df_m1.index.min()}  →  {df_m1.index.max()}")
df_m1.head(3)

## 2 — Run Pipeline

In [ ]:
pipeline = ForexPipeline(
    lags            = DATA_CFG["lags"],
    target_horizons = DATA_CFG["target_horizons"],
    gap_bars        = DATA_CFG["gap_bars"],
    scaling         = DATA_CFG["scaling"],
    window_size     = DATA_CFG["window_size"],
    norm_method     = DATA_CFG["norm_method"],
    fracdiff_d      = DATA_CFG["fracdiff_d"],
    target_type     = DATA_CFG["target_type"],
    k_up            = DATA_CFG["k_up"],
    k_down          = DATA_CFG["k_down"],
    horizon_bars    = DATA_CFG["horizon_bars"],
    barrier_price   = DATA_CFG.get("barrier_price", "hl"),
    barrier_norm_method = DATA_CFG.get("barrier_norm_method", "raw"),
    threshold       = DATA_CFG["threshold"],
    weekends        = DATA_CFG["weekends"],
)

results      = pipeline.run(df_m1, timeframe=DATA_CFG["timeframe"])
feature_cols = results["feature_cols"]
target_col   = DATA_CFG["target_col"]

for split in ["train", "val", "test"]:
    df_s = results[split]
    print(f"{split:5s}: {len(df_s):6d} rows  "
          f"{df_s.index.min().date()} → {df_s.index.max().date()}")

print(f"\nPipeline features ({len(feature_cols)}): {feature_cols}")


## 3 — Load & Join NN Predictor Features


In [ ]:
import re as _re


def _pname(p: float, prefix: str = "q") -> str:
    return f"{prefix}{int(round(p * 100)):02d}"


def _select_source_columns(source: str, raw: pd.DataFrame, cfg: dict) -> list[str]:
    horizons = sorted(set(cfg["horizons"]))
    wanted_h = set(horizons)

    if source == "kronos" and cfg.get("n_samples", 1) == 1:
        expected = [
            f"{name}_h{h}"
            for h in horizons
            for name in ["close", "high", "low", "spread"]
        ]
    else:
        prefix = "tfm_q" if source == "timesfm" else "q"
        expected = [
            f"{_pname(p, prefix)}_h{h}"
            for h in horizons
            for p in sorted(set(cfg["percentiles"]))
        ]

    missing = [col for col in expected if col not in raw.columns]
    if missing:
        raise KeyError(
            f"{source} parquet is missing expected columns: {missing}. "
            f"Available columns: {list(raw.columns)}"
        )

    selected = expected.copy()
    if cfg.get("use_staleness", False):
        if "staleness" not in raw.columns:
            raise KeyError(f"{source} parquet does not contain staleness.")
        selected.append("staleness")

    leaked = [col for col in selected if col == "run_id"]
    assert not leaked, "run_id should remain metadata and not enter model features."
    return selected


nn_feature_frames = {}
nn_feature_cols_by_source = {}
nn_feature_cols = []
nn_feature_provider_by_col = {}

for source, cfg in NN_FEATURES_CFG.items():
    if not cfg.get("enabled", False):
        continue

    raw = pd.read_parquet(nn_feature_paths[source]["path"])
    source_cols = _select_source_columns(source, raw, cfg)
    prefixed = raw[source_cols].rename(columns={col: f"{source}_{col}" for col in source_cols})

    nn_feature_frames[source] = prefixed
    nn_feature_cols_by_source[source] = list(prefixed.columns)
    nn_feature_cols.extend(prefixed.columns)
    nn_feature_provider_by_col.update({col: source for col in prefixed.columns})

    print(f"{source:8s} selected columns ({len(source_cols)}): {source_cols}")
    print(f"{source:8s} prefixed columns ({len(prefixed.columns)}): {list(prefixed.columns)}")

if len(nn_feature_cols) != len(set(nn_feature_cols)):
    duplicates = pd.Series(nn_feature_cols).value_counts()
    raise ValueError(f"Duplicate NN feature names after prefixing: {duplicates[duplicates > 1].to_dict()}")

splits_extended = {}
for split_name in ["train", "val", "test"]:
    df = results[split_name].copy()
    n_before = len(df)
    for source, frame in nn_feature_frames.items():
        df = df.join(frame, how="left")
    df = df.dropna(subset=nn_feature_cols)
    n_dropped = n_before - len(df)
    splits_extended[split_name] = df
    print(f"{split_name:5s}: {len(df):6d} rows  "
          f"({n_dropped} dropped — no selected NN feature coverage)  "
          f"{df.index.min().date()} -> {df.index.max().date()}")

feature_cols_ext = feature_cols + nn_feature_cols
ENABLED_NN_LABEL = "+".join(nn_feature_frames.keys()).upper()
print(f"\nExtended feature set ({len(feature_cols_ext)}): "
      f"{len(feature_cols)} pipeline + {len(nn_feature_cols)} NN")
for source, cols in nn_feature_cols_by_source.items():
    print(f"  {source:8s}: {len(cols)} features")
print(f"Enabled NN label: {ENABLED_NN_LABEL}")


## 4 — Extract Binary Short / Long X / y Arrays


In [ ]:
# Binary class mapping after filtering triple_barrier labels:
#   0 = short hit (tb_label = -1 : price hit the lower barrier)
#   1 = long hit  (tb_label = +1 : price hit the upper barrier)
# Neutral rows (tb_label = 0) are dropped from each split.
CLASS_LABELS = {0: "short", 1: "long"}


def extract_xy(splits_ext, split, target_col, feature_cols_ext):
    X, y_raw = pipeline.get_xy(splits_ext[split], target_col, feature_cols_ext)
    if target_col != "tb_label":
        raise ValueError("Binary short/long notebook expects DATA_CFG['target_col'] == 'tb_label'.")

    decisive_mask = y_raw != 0
    X = X[decisive_mask]
    y = (y_raw[decisive_mask] == 1).astype(int)  # -1 -> 0 short, +1 -> 1 long

    if len(y) == 0:
        raise ValueError(f"{split} split has no short/long samples after dropping neutral labels.")
    return X, y


X_train, y_train = extract_xy(splits_extended, "train", target_col, feature_cols_ext)
X_val,   y_val   = extract_xy(splits_extended, "val",   target_col, feature_cols_ext)
X_test,  y_test  = extract_xy(splits_extended, "test",  target_col, feature_cols_ext)

print("Binary class mapping:")
for cls, lbl in CLASS_LABELS.items():
    print(f"  {cls} = {lbl}")
print()

for name, y in [("train", y_train), ("val", y_val), ("test", y_test)]:
    total = len(y)
    unique, counts = np.unique(y, return_counts=True)
    dist_str = "  ".join(
        f"{CLASS_LABELS[int(k)]}({int(k)}): {int(v):,} ({int(v)/total:.1%})"
        for k, v in zip(unique, counts)
    )
    print(f"{name:5s}: {total:,} decisive samples  |  {dist_str}")


## 5 — Train Binary XGBoost + Log Everything to MLflow


In [ ]:
mlflow.set_tracking_uri(MLFLOW_DB)
mlflow.set_experiment(EXPERIMENT_NAME)

run_name = (
    f"{DATA_CFG['pair']}_{DATA_CFG['timeframe']}_"
    f"{DATA_CFG['norm_method']}_binary_short_long_"
    f"{ENABLED_NN_LABEL.lower()}"
)

with mlflow.start_run(run_name=run_name) as run:
    RUN_ID = run.info.run_id

    # ── Parameters ───────────────────────────────────────────────────────────
    mlflow.log_params({f"data_{k}": str(v) for k, v in DATA_CFG.items()})
    mlflow.log_params({f"xgb_{k}": v for k, v in XGB_PARAMS.items()})
    mlflow.log_param("label_mode", "drop_neutral_short0_long1")
    mlflow.log_params({f"conf_{k}": str(v) for k, v in CONFIDENCE_CFG.items()})
    mlflow.log_params({f"diag_{k}": str(v) for k, v in DIAGNOSTIC_CFG.items()})

    for source, cfg in NN_FEATURES_CFG.items():
        if not cfg.get("enabled", False):
            mlflow.log_param(f"{source}_enabled", False)
            continue
        mlflow.log_params({f"{source}_{k}": str(v) for k, v in cfg.items()})
        mlflow.log_param(f"{source}_file", nn_feature_paths[source]["fname"])
        mlflow.log_param(f"n_{source}_features", len(nn_feature_cols_by_source[source]))

    mlflow.log_param("enabled_nn_sources", ",".join(nn_feature_frames.keys()))
    mlflow.log_param("n_pipeline_features", len(feature_cols))
    mlflow.log_param("n_nn_features", len(nn_feature_cols))
    mlflow.log_param("n_features_total", len(feature_cols_ext))

    mlflow.log_params({
        "raw_m1_start":  str(df_m1.index.min()),
        "raw_m1_end":    str(df_m1.index.max()),
        "train_start":   str(splits_extended["train"].index.min()),
        "train_end":     str(splits_extended["train"].index.max()),
        "val_start":     str(splits_extended["val"].index.min()),
        "val_end":       str(splits_extended["val"].index.max()),
        "test_start":    str(splits_extended["test"].index.min()),
        "test_end":      str(splits_extended["test"].index.max()),
        "train_samples": len(X_train),
        "val_samples":   len(X_val),
        "test_samples":  len(X_test),
        "train_long_rate": float(y_train.mean()),
        "val_long_rate":   float(y_val.mean()),
        "test_long_rate":  float(y_test.mean()),
    })

    mlflow.log_dict({
        "pipeline_features": feature_cols,
        "nn_features_by_source": nn_feature_cols_by_source,
        "features": feature_cols_ext,
    }, "features.json")

    # ── Train ─────────────────────────────────────────────────────────────────
    model = xgb.XGBClassifier(**XGB_PARAMS)
    model.fit(
        X_train, y_train,
        eval_set=[(X_train, y_train), (X_val, y_val)],
        verbose=False,
    )

    mlflow.log_param("best_iteration", model.best_iteration)
    mlflow.xgboost.log_model(model, "model")

    # ── Metrics per split ─────────────────────────────────────────────────────
    # proba shape: (n, 2) -> [P(short/0), P(long/1)]
    split_metrics = {}
    for sname, X, y in [("train", X_train, y_train),
                         ("val",   X_val,   y_val),
                         ("test",  X_test,  y_test)]:
        proba      = model.predict_proba(X)
        proba_long = proba[:, 1]
        pred       = model.predict(X)
        m = {
            "auc":           roc_auc_score(y, proba_long),
            "avg_precision": average_precision_score(y, proba_long),
            "logloss":       log_loss(y, proba),
            "brier":         brier_score_loss(y, proba_long),
            "f1":            f1_score(y, pred, zero_division=0),
            "precision":     precision_score(y, pred, zero_division=0),
            "recall":        recall_score(y, pred, zero_division=0),
            "balanced_acc":  balanced_accuracy_score(y, pred),
            "mcc":           matthews_corrcoef(y, pred),
        }
        split_metrics[sname] = m
        mlflow.log_metrics({f"{sname}_{k}": v for k, v in m.items()})

    print(f"Run ID   : {RUN_ID}")
    print(f"Best iter: {model.best_iteration}")
    print(pd.DataFrame(split_metrics).T.round(4))


## 6 — Analysis: Calibration and Probability Distribution Diagnostics


In [ ]:
colors = {"train": "steelblue", "val": "darkorange", "test": "seagreen"}
styles = {"train": "--",        "val": "-",          "test": "-"}
_eval_data = {
    "train": (X_train, y_train),
    "val":   (X_val,   y_val),
    "test":  (X_test,  y_test),
}
_calibration_splits = ["train", "val", "test"]
_n_bins = int(DIAGNOSTIC_CFG.get("calibration_bins", 10))

fig, axes = plt.subplots(2, 2, figsize=(13, 10))
ax_uniform, ax_quantile, ax_hist, ax_counts = axes.ravel()
ax_uniform.plot([0, 1], [0, 1], 'k--', lw=1, label="perfect calibration")
ax_quantile.plot([0, 1], [0, 1], 'k--', lw=1, label="perfect calibration")

probability_summary_rows = []
brier_scores = {}

for sname in _calibration_splits:
    X, y = _eval_data[sname]
    proba_long = model.predict_proba(X)[:, 1]
    brier = brier_score_loss(y, proba_long)
    brier_scores[sname] = brier

    frac_u, mean_u = calibration_curve(y, proba_long, n_bins=_n_bins, strategy="uniform")
    frac_q, mean_q = calibration_curve(y, proba_long, n_bins=_n_bins, strategy="quantile")
    ax_uniform.plot(mean_u, frac_u, 'o-', color=colors[sname], lw=2,
                    label=f"{sname} Brier={brier:.4f} bins={len(mean_u)}")
    ax_quantile.plot(mean_q, frac_q, 'o-', color=colors[sname], lw=2,
                     label=f"{sname} Brier={brier:.4f} bins={len(mean_q)}")
    ax_hist.hist(proba_long, bins=np.linspace(0, 1, _n_bins + 1), alpha=0.35,
                 color=colors[sname], label=sname)

    uniform_counts, uniform_edges = np.histogram(proba_long, bins=np.linspace(0, 1, _n_bins + 1))
    quantiles = np.quantile(proba_long, [0, 0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99, 1.0])
    probability_summary_rows.append({
        "split": sname,
        "n": len(y),
        "long_rate": float(y.mean()),
        "p_min": float(quantiles[0]),
        "p01": float(quantiles[1]),
        "p05": float(quantiles[2]),
        "p25": float(quantiles[3]),
        "p50": float(quantiles[4]),
        "p75": float(quantiles[5]),
        "p95": float(quantiles[6]),
        "p99": float(quantiles[7]),
        "p_max": float(quantiles[8]),
        "uniform_nonempty_bins": int((uniform_counts > 0).sum()),
        "uniform_bin_counts": uniform_counts.tolist(),
        "uniform_bin_edges": uniform_edges.round(3).tolist(),
    })

probability_summary_df = pd.DataFrame(probability_summary_rows).set_index("split")
print("Probability distribution summary:")
print(probability_summary_df.drop(columns=["uniform_bin_counts", "uniform_bin_edges"]).round(4).to_string())
print("\nUniform calibration bin counts:")
for split, row in probability_summary_df.iterrows():
    print(f"{split:5s}: {row['uniform_bin_counts']}")

ax_uniform.set_title("Uniform-Bin Calibration")
ax_uniform.set_xlabel("Mean predicted P(long)")
ax_uniform.set_ylabel("Actual long rate")
ax_uniform.legend(fontsize=8)

ax_quantile.set_title("Quantile-Bin Calibration")
ax_quantile.set_xlabel("Mean predicted P(long)")
ax_quantile.set_ylabel("Actual long rate")
ax_quantile.legend(fontsize=8)

ax_hist.set_title("Predicted P(long) Histogram")
ax_hist.set_xlabel("P(long)")
ax_hist.set_ylabel("Count")
ax_hist.legend(fontsize=8)

_count_df = probability_summary_df[["uniform_nonempty_bins"]]
_count_df.plot.bar(ax=ax_counts, rot=0, color="slategray", legend=False)
ax_counts.set_title("Non-Empty Uniform Calibration Bins")
ax_counts.set_ylabel(f"out of {_n_bins}")
ax_counts.set_ylim(0, _n_bins)

plt.suptitle(f"Calibration Diagnostics — {DATA_CFG['pair']} {DATA_CFG['timeframe']} + {ENABLED_NN_LABEL}", y=1.02)
plt.tight_layout()

with mlflow.start_run(run_id=RUN_ID):
    mlflow.log_metrics({f"{k}_brier": v for k, v in brier_scores.items()})
    with tempfile.TemporaryDirectory() as tmp:
        fig_path = os.path.join(tmp, "calibration_curve.png")
        csv_path = os.path.join(tmp, "probability_distribution.csv")
        fig.savefig(fig_path)
        probability_summary_df.to_csv(csv_path)
        mlflow.log_artifact(fig_path, "plots")
        mlflow.log_artifact(csv_path, "reports")

plt.show()


## 7 — Threshold Sweep and Abstention Diagnostics


In [ ]:
threshold_splits = DIAGNOSTIC_CFG.get("threshold_splits", ["val", "test"])
thresholds = [float(t) for t in DIAGNOSTIC_CFG.get("thresholds", [])]
min_coverage_warn = float(DIAGNOSTIC_CFG.get("min_coverage_warn", 0.01))
near_random_warn = float(DIAGNOSTIC_CFG.get("near_random_accuracy_warn", 0.53))

if not thresholds:
    raise ValueError("DIAGNOSTIC_CFG['thresholds'] must contain at least one threshold.")

threshold_rows = []
threshold_arrays = {}

for sname in threshold_splits:
    if sname not in _eval_data:
        raise ValueError(f"Unknown threshold split: {sname!r}")
    X, y = _eval_data[sname]
    proba_long = model.predict_proba(X)[:, 1]
    proba_short = 1 - proba_long

    for t in thresholds:
        selectors = {
            "long_only": proba_long >= t,
            "short_only": proba_short >= t,
            "either_side": (proba_long >= t) | (proba_short >= t),
        }

        for mode, mask in selectors.items():
            n = int(mask.sum())
            row = {
                "split": sname,
                "mode": mode,
                "threshold": t,
                "n_selected": n,
                "coverage": n / len(y) if len(y) else np.nan,
            }

            if n:
                if mode == "long_only":
                    pred_sel = np.ones(n, dtype=int)
                    y_side = y[mask]
                    proba_side = proba_long[mask]
                elif mode == "short_only":
                    pred_sel = np.zeros(n, dtype=int)
                    y_side = y[mask]
                    proba_side = proba_short[mask]
                else:
                    pred_sel = (proba_long[mask] >= t).astype(int)
                    y_side = y[mask]
                    proba_side = np.maximum(proba_long[mask], proba_short[mask])

                row.update({
                    "accuracy": float((pred_sel == y_side).mean()),
                    "actual_long_rate": float(y_side.mean()),
                    "actual_short_rate": float(1 - y_side.mean()),
                    "pred_long_rate": float(pred_sel.mean()),
                    "avg_confidence": float(proba_side.mean()),
                    "precision_long": float(precision_score(y_side, pred_sel, zero_division=0)),
                    "recall_long": float(recall_score(y_side, pred_sel, zero_division=0)),
                    "f1_long": float(f1_score(y_side, pred_sel, zero_division=0)),
                    "mcc": float(matthews_corrcoef(y_side, pred_sel)) if len(np.unique(y_side)) > 1 and len(np.unique(pred_sel)) > 1 else np.nan,
                })
                threshold_arrays[(sname, mode, t)] = {"y": y_side, "pred": pred_sel}
            else:
                row.update({
                    "accuracy": np.nan,
                    "actual_long_rate": np.nan,
                    "actual_short_rate": np.nan,
                    "pred_long_rate": np.nan,
                    "avg_confidence": np.nan,
                    "precision_long": np.nan,
                    "recall_long": np.nan,
                    "f1_long": np.nan,
                    "mcc": np.nan,
                })

            threshold_rows.append(row)

threshold_df = pd.DataFrame(threshold_rows)
print("Threshold sweep summary:")
_display_cols = ["split", "mode", "threshold", "n_selected", "coverage", "accuracy", "avg_confidence", "actual_long_rate", "pred_long_rate", "precision_long", "recall_long", "f1_long"]
print(threshold_df[_display_cols].round(4).to_string(index=False))

usable = threshold_df[
    (threshold_df["coverage"] >= min_coverage_warn)
    & (threshold_df["accuracy"] >= near_random_warn)
].copy()
if usable.empty:
    print(f"\nWARNING: no threshold has coverage >= {min_coverage_warn:.1%} and accuracy >= {near_random_warn:.2f}.")
    print("The model does not currently expose a reliable high-confidence region on these splits.")
else:
    print("\nPotentially usable threshold rows:")
    print(usable[_display_cols].sort_values(["split", "mode", "threshold"]).round(4).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for mode in ["long_only", "short_only", "either_side"]:
    for sname in threshold_splits:
        dfp = threshold_df[(threshold_df["split"] == sname) & (threshold_df["mode"] == mode)]
        label = f"{sname} {mode}"
        axes[0].plot(dfp["threshold"], dfp["coverage"], marker='o', label=label)
        axes[1].plot(dfp["threshold"], dfp["accuracy"], marker='o', label=label)
axes[0].axhline(min_coverage_warn, color='gray', ls=':', lw=1, label="min coverage warn")
axes[0].set_title("Coverage by Threshold")
axes[0].set_xlabel("Confidence threshold")
axes[0].set_ylabel("Coverage")
axes[1].axhline(0.5, color='black', ls=':', lw=1, label="random")
axes[1].axhline(near_random_warn, color='gray', ls='--', lw=1, label="near-random warn")
axes[1].set_title("Accuracy by Threshold")
axes[1].set_xlabel("Confidence threshold")
axes[1].set_ylabel("Accuracy on selected predictions")
for ax in axes:
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=7)
plt.tight_layout()

with mlflow.start_run(run_id=RUN_ID):
    with tempfile.TemporaryDirectory() as tmp:
        csv_path = os.path.join(tmp, "threshold_sweep.csv")
        fig_path = os.path.join(tmp, "threshold_sweep.png")
        threshold_df.to_csv(csv_path, index=False)
        fig.savefig(fig_path)
        mlflow.log_artifact(csv_path, "reports")
        mlflow.log_artifact(fig_path, "plots")
plt.show()


## 8 — Confidence-Band Prediction Quality


In [ ]:
conf_splits = CONFIDENCE_CFG.get("splits", ["val", "test"])
confidence_bands = CONFIDENCE_CFG.get("bands", [])

if not confidence_bands:
    raise ValueError("CONFIDENCE_CFG['bands'] must contain at least one confidence band.")

confidence_arrays = {}
confidence_rows = []

for band_idx, band in enumerate(confidence_bands):
    side = band.get("side")
    if side not in {"long", "short"}:
        raise ValueError(f"Confidence band {band_idx} has invalid side={side!r}; expected 'long' or 'short'.")

    conf_min = float(band["min_p"])
    conf_max = float(band["max_p"])
    if not 0 <= conf_min <= conf_max <= 1:
        raise ValueError(f"Confidence band {band_idx} must satisfy 0 <= min_p <= max_p <= 1.")

    band_name = band.get("name") or f"{side}_{conf_min:.2f}_{conf_max:.2f}".replace(".", "")
    target_class = 1 if side == "long" else 0
    positive_label = CLASS_LABELS[target_class]

    for sname in conf_splits:
        if sname not in _eval_data:
            raise ValueError(f"Unknown confidence split: {sname!r}")

        X, y = _eval_data[sname]
        proba = model.predict_proba(X)
        proba_side = proba[:, target_class]
        pred = model.predict(X)
        pred_side = (pred == target_class).astype(int)
        y_side = (y == target_class).astype(int)
        mask = (proba_side >= conf_min) & (proba_side <= conf_max)

        y_f = y[mask]
        pred_f = pred[mask]
        proba_f = proba_side[mask]
        y_side_f = y_side[mask]
        pred_side_f = pred_side[mask]
        total = len(y)
        n = int(mask.sum())

        row = {
            "band": band_name,
            "side": side,
            "split": sname,
            "target_class": target_class,
            "n_total": total,
            "n_in_band": n,
            "coverage": n / total if total else np.nan,
            "p_min": conf_min,
            "p_max": conf_max,
        }

        if n:
            proba_matrix = np.column_stack([1 - proba_f, proba_f])
            row.update({
                f"actual_{side}_rate": float(y_side_f.mean()),
                f"predicted_{side}_rate": float(pred_side_f.mean()),
                f"avg_p_{side}": float(proba_f.mean()),
                f"min_p_{side}": float(proba_f.min()),
                f"max_p_{side}": float(proba_f.max()),
                "accuracy": float((pred_f == y_f).mean()),
                "precision": float(precision_score(y_side_f, pred_side_f, zero_division=0)),
                "recall": float(recall_score(y_side_f, pred_side_f, zero_division=0)),
                "f1": float(f1_score(y_side_f, pred_side_f, zero_division=0)),
                "brier": float(brier_score_loss(y_side_f, proba_f)),
                "logloss": float(log_loss(y_side_f, proba_matrix, labels=[0, 1])),
                "true_short": int((y_f == 0).sum()),
                "true_long": int((y_f == 1).sum()),
                "pred_short": int((pred_f == 0).sum()),
                "pred_long": int((pred_f == 1).sum()),
            })
        else:
            row.update({
                f"actual_{side}_rate": np.nan,
                f"predicted_{side}_rate": np.nan,
                f"avg_p_{side}": np.nan,
                f"min_p_{side}": np.nan,
                f"max_p_{side}": np.nan,
                "accuracy": np.nan,
                "precision": np.nan,
                "recall": np.nan,
                "f1": np.nan,
                "brier": np.nan,
                "logloss": np.nan,
                "true_short": 0,
                "true_long": 0,
                "pred_short": 0,
                "pred_long": 0,
            })
            print(f"{band_name}/{sname}: no predictions in confidence band [{conf_min:.2f}, {conf_max:.2f}] for P({positive_label}).")

        confidence_rows.append(row)
        confidence_arrays[(band_name, sname)] = {
            "mask": mask,
            "side": side,
            "target_class": target_class,
            "y": y_f,
            "pred": pred_f,
            "y_side": y_side_f,
            "pred_side": pred_side_f,
            "proba_side": proba_f,
        }

confidence_df = pd.DataFrame(confidence_rows).set_index(["band", "split"])
print("Confidence bands:")
for band in confidence_bands:
    print(f"  {band.get('name')}: {band['min_p']:.2f} <= P({band['side']}) <= {band['max_p']:.2f}")
print()
print(confidence_df.round(4).to_string())

print("\nConfidence-band warnings:")
_warning_found = False
for (band_name, sname), row in confidence_df.iterrows():
    coverage = row.get("coverage", np.nan)
    accuracy = row.get("accuracy", np.nan)
    if pd.notna(coverage) and coverage < DIAGNOSTIC_CFG.get("min_coverage_warn", 0.01):
        print(f"  {band_name}/{sname}: low coverage {coverage:.2%}; confidence band may be too high for this model.")
        _warning_found = True
    if pd.notna(accuracy) and accuracy < DIAGNOSTIC_CFG.get("near_random_accuracy_warn", 0.53):
        print(f"  {band_name}/{sname}: selected accuracy {accuracy:.3f} is near random.")
        _warning_found = True
if not _warning_found:
    print("  none")

with mlflow.start_run(run_id=RUN_ID):
    for (band_name, sname), row in confidence_df.iterrows():
        metrics = {}
        safe_band = str(band_name).replace(" ", "_")
        for key, val in row.items():
            if key in {"side", "p_min", "p_max"}:
                continue
            if pd.notna(val) and isinstance(val, (int, float, np.integer, np.floating)):
                metrics[f"conf_band_{safe_band}_{sname}_{key}"] = float(val)
        if metrics:
            mlflow.log_metrics(metrics)

    with tempfile.TemporaryDirectory() as tmp:
        csv_path = os.path.join(tmp, "confidence_band_metrics.csv")
        confidence_df.to_csv(csv_path)
        mlflow.log_artifact(csv_path, "reports")


## 9 — Confidence-Band Plots


In [ ]:
plot_df = confidence_df.reset_index()
plot_df["label"] = plot_df["band"] + " / " + plot_df["split"]
plot_colors = [colors.get(s, "gray") for s in plot_df["split"]]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_df.plot.bar(x="label", y="n_in_band", ax=axes[0], color=plot_colors, rot=35, legend=False)
axes[0].set_title("Predictions in Confidence Bands")
axes[0].set_ylabel("Count")
axes[0].set_xlabel("")
plot_df.plot.bar(x="label", y="coverage", ax=axes[1], color=plot_colors, rot=35, legend=False)
axes[1].set_title("Coverage")
axes[1].set_ylabel("Share of split")
axes[1].set_xlabel("")
coverage_max = plot_df["coverage"].max(skipna=True)
axes[1].set_ylim(0, max(0.05, min(1.0, float(coverage_max * 1.2) if pd.notna(coverage_max) else 0.05)))
plt.suptitle("Confidence-Band Sample Counts", y=1.04)
plt.tight_layout()

with mlflow.start_run(run_id=RUN_ID):
    with tempfile.TemporaryDirectory() as tmp:
        path = os.path.join(tmp, "confidence_band_counts.png")
        fig.savefig(path)
        mlflow.log_artifact(path, "plots")
plt.show()

quality_cols = ["accuracy", "precision", "recall", "f1"]
fig2, ax2 = plt.subplots(figsize=(12, 5))
plot_df.set_index("label")[quality_cols].plot.bar(ax=ax2, rot=35)
ax2.set_title("Prediction Quality Inside Confidence Bands")
ax2.set_ylim(0, 1.05)
ax2.legend(fontsize=8)
plt.tight_layout()

with mlflow.start_run(run_id=RUN_ID):
    with tempfile.TemporaryDirectory() as tmp:
        path = os.path.join(tmp, "confidence_band_quality.png")
        fig2.savefig(path)
        mlflow.log_artifact(path, "plots")
plt.show()

fig3, ax3 = plt.subplots(figsize=(8, 6))
ax3.plot([0, 1], [0, 1], 'k--', lw=1, label="perfect calibration")
for _, row in plot_df.iterrows():
    side = row["side"]
    avg_col = f"avg_p_{side}"
    actual_col = f"actual_{side}_rate"
    if avg_col in row and actual_col in row and pd.notna(row[avg_col]) and pd.notna(row[actual_col]):
        ax3.scatter(row[avg_col], row[actual_col], s=90,
                    color=colors.get(row["split"], "gray"),
                    marker="o" if side == "long" else "s",
                    label=f"{row['band']} / {row['split']} (n={int(row['n_in_band'])})")
        ax3.vlines(row[avg_col], min(row[avg_col], row[actual_col]),
                   max(row[avg_col], row[actual_col]),
                   color=colors.get(row["split"], "gray"), alpha=0.5)
ax3.set_xlim(0, 1)
ax3.set_ylim(0, 1)
ax3.set_xlabel("Average predicted probability for configured side")
ax3.set_ylabel("Actual hit rate for configured side")
ax3.set_title("Focused Calibration Inside Confidence Bands")
ax3.legend(fontsize=7)
plt.tight_layout()

with mlflow.start_run(run_id=RUN_ID):
    with tempfile.TemporaryDirectory() as tmp:
        path = os.path.join(tmp, "confidence_band_calibration.png")
        fig3.savefig(path)
        mlflow.log_artifact(path, "plots")
plt.show()

n_panels = len(plot_df)
n_cols = min(3, max(1, n_panels))
n_rows = int(np.ceil(n_panels / n_cols))
fig4, axes4 = plt.subplots(n_rows, n_cols, figsize=(5.5 * n_cols, 4.5 * n_rows), squeeze=False)
cm_labels = [f"{CLASS_LABELS[i]} ({i})" for i in range(2)]
for ax, (_, row) in zip(axes4.ravel(), plot_df.iterrows()):
    key = (row["band"], row["split"])
    y_f = confidence_arrays[key]["y"]
    pred_f = confidence_arrays[key]["pred"]
    if len(y_f) == 0:
        ax.axis("off")
        ax.text(0.5, 0.5, f"{row['band']} / {row['split']}: no samples", ha="center", va="center")
        continue
    cm = confusion_matrix(y_f, pred_f, labels=[0, 1])
    row_sum = cm.sum(axis=1, keepdims=True)
    cm_norm = np.divide(cm.astype(float), row_sum, out=np.zeros_like(cm, dtype=float), where=row_sum != 0)
    annot = np.array([[f"{cm[i,j]}\n({cm_norm[i,j]:.1%})" for j in range(2)] for i in range(2)])
    sns.heatmap(cm_norm, annot=annot, fmt='', cmap='Blues', ax=ax,
                xticklabels=[f"pred {l}" for l in cm_labels],
                yticklabels=[f"true {l}" for l in cm_labels],
                vmin=0, vmax=1, linewidths=0.5)
    ax.set_title(f"{row['band']} / {row['split']}")
for ax in axes4.ravel()[n_panels:]:
    ax.axis("off")
plt.tight_layout()

with mlflow.start_run(run_id=RUN_ID):
    with tempfile.TemporaryDirectory() as tmp:
        path = os.path.join(tmp, "confidence_band_confusion_matrices.png")
        fig4.savefig(path)
        mlflow.log_artifact(path, "plots")
plt.show()


## 10 — Analysis: ROC Curves (train / val / test)


In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

for sname, X, y in [("train", X_train, y_train),
                     ("val",   X_val,   y_val),
                     ("test",  X_test,  y_test)]:
    proba_long = model.predict_proba(X)[:, 1]
    fpr, tpr, _ = roc_curve(y, proba_long)
    auc        = roc_auc_score(y, proba_long)
    ax.plot(fpr, tpr, ls=styles[sname], color=colors[sname],
            label=f"{sname}  AUC = {auc:.4f}", lw=2)

ax.plot([0, 1], [0, 1], 'k:', lw=1, label="random")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title(f"ROC Curve (long) — {DATA_CFG['pair']} {DATA_CFG['timeframe']} + {ENABLED_NN_LABEL}")
ax.legend(loc="lower right")
plt.tight_layout()

with mlflow.start_run(run_id=RUN_ID):
    with tempfile.TemporaryDirectory() as tmp:
        path = os.path.join(tmp, "roc_curve.png")
        fig.savefig(path)
        mlflow.log_artifact(path, "plots")

plt.show()


## 11 — Analysis: Precision-Recall Curves


In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

for sname, X, y in [("val",  X_val,  y_val),
                     ("test", X_test, y_test)]:
    proba_long = model.predict_proba(X)[:, 1]
    prec, rec, _ = precision_recall_curve(y, proba_long)
    ap         = average_precision_score(y, proba_long)
    ax.plot(rec, prec, color=colors[sname],
            label=f"{sname}  AP = {ap:.4f}", lw=2)

for sname, y in [("val", y_val), ("test", y_test)]:
    ax.axhline(y.mean(), ls=':', color=colors[sname], alpha=0.5)

ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title(f"Precision-Recall (long) — {DATA_CFG['pair']} {DATA_CFG['timeframe']} + {ENABLED_NN_LABEL}")
ax.legend()
plt.tight_layout()

with mlflow.start_run(run_id=RUN_ID):
    with tempfile.TemporaryDirectory() as tmp:
        path = os.path.join(tmp, "pr_curve.png")
        fig.savefig(path)
        mlflow.log_artifact(path, "plots")

plt.show()


## 12 — Analysis: Confusion Matrices (val + test)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

cm_labels = [f"{CLASS_LABELS[i]} ({i})" for i in range(2)]

for ax, (sname, X, y) in zip(axes, [("val", X_val, y_val), ("test", X_test, y_test)]):
    pred    = model.predict(X)
    cm      = confusion_matrix(y, pred, labels=[0, 1])
    row_sum = cm.sum(axis=1, keepdims=True)
    cm_norm = np.divide(cm.astype(float), row_sum, out=np.zeros_like(cm, dtype=float), where=row_sum != 0)
    annot   = np.array([[f"{cm[i,j]}\n({cm_norm[i,j]:.1%})" for j in range(2)] for i in range(2)])
    sns.heatmap(cm_norm, annot=annot, fmt='', cmap='Blues', ax=ax,
                xticklabels=[f"pred {l}" for l in cm_labels],
                yticklabels=[f"true {l}" for l in cm_labels],
                vmin=0, vmax=1, linewidths=0.5)
    ax.set_title(f"Confusion Matrix — {sname}")

plt.tight_layout()

with mlflow.start_run(run_id=RUN_ID):
    with tempfile.TemporaryDirectory() as tmp:
        path = os.path.join(tmp, "confusion_matrices.png")
        fig.savefig(path)
        mlflow.log_artifact(path, "plots")

plt.show()


## 13 — Analysis: Classification Report


In [ ]:
report_frames = []
_target_names = [CLASS_LABELS[i] for i in range(2)]

for sname, X, y in [("val", X_val, y_val), ("test", X_test, y_test)]:
    pred = model.predict(X)

    total = len(y)
    unique, counts = np.unique(y, return_counts=True)
    dist = {CLASS_LABELS[int(k)]: int(v) for k, v in zip(unique, counts)}
    print(f"\n-- {sname}  (n={total:,}) --")
    print("  Class counts: " + "  ".join(f"{lbl}: {cnt:,} ({cnt/total:.1%})"
                                          for lbl, cnt in dist.items()))

    print(classification_report(y, pred, labels=[0, 1], target_names=_target_names, zero_division=0))

    rpt    = classification_report(y, pred, labels=[0, 1], target_names=_target_names, output_dict=True, zero_division=0)
    df_rpt = pd.DataFrame(rpt).T.round(4)
    df_rpt.insert(0, "split", sname)
    report_frames.append(df_rpt)

combined_report = pd.concat(report_frames)

with mlflow.start_run(run_id=RUN_ID):
    with tempfile.TemporaryDirectory() as tmp:
        path = os.path.join(tmp, "classification_report.csv")
        combined_report.to_csv(path)
        mlflow.log_artifact(path, "reports")


## 14 — Binary Class Prediction Quality


In [ ]:
_classes = list(CLASS_LABELS.keys())
_splits  = [("val", X_val, y_val), ("test", X_test, y_test)]
_clr     = {"val": "steelblue", "test": "darkorange"}

_x      = np.arange(len(_classes))
_w      = 0.35
_labels = [f"{CLASS_LABELS[c]}\n({c})" for c in _classes]

fig, axes = plt.subplots(1, 3, figsize=(14, 5), sharey=True)
for ax, metric_fn, title in zip(
    axes,
    [precision_score, recall_score, f1_score],
    ["Precision", "Recall", "F1"],
):
    for offset, (sname, X, y) in zip([-_w / 2, _w / 2], _splits):
        scores = metric_fn(y, model.predict(X), labels=_classes, average=None, zero_division=0)
        ax.bar(_x + offset, scores, _w, label=sname, color=_clr[sname], alpha=0.85)
    ax.set_xticks(_x)
    ax.set_xticklabels(_labels)
    ax.set_ylim(0, 1.05)
    ax.set_title(title)
    ax.legend(fontsize=9)

fig.suptitle(f"Binary Precision / Recall / F1  (val vs test) + {ENABLED_NN_LABEL}", fontsize=13)
plt.tight_layout()

with mlflow.start_run(run_id=RUN_ID):
    with tempfile.TemporaryDirectory() as tmp:
        _p = os.path.join(tmp, "binary_prf1.png")
        fig.savefig(_p)
        mlflow.log_artifact(_p, "plots")
plt.show()


## 15 — Analysis: Learning Curves (Overfitting Diagnostic)


In [ ]:
evals = model.evals_result()
train_logloss = evals["validation_0"]["logloss"]
val_logloss   = evals["validation_1"]["logloss"]
rounds = range(1, len(train_logloss) + 1)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(rounds, train_logloss, label="train logloss", color="steelblue", lw=1.5)
ax.plot(rounds, val_logloss,   label="val logloss",   color="darkorange", lw=1.5)
ax.axvline(model.best_iteration, ls='--', color='seagreen', lw=1.5,
           label=f"best iter = {model.best_iteration}")

if model.best_iteration < len(train_logloss):
    ax.axvspan(model.best_iteration, len(train_logloss), alpha=0.08, color='red',
               label="overfitting region")

ax.set_xlabel("Boosting Round")
ax.set_ylabel("Log Loss")
ax.set_title(f"Learning Curves — {DATA_CFG['pair']} {DATA_CFG['timeframe']} + {ENABLED_NN_LABEL}")
ax.legend()
plt.tight_layout()

overfit_gap = val_logloss[model.best_iteration - 1] - train_logloss[model.best_iteration - 1]
with mlflow.start_run(run_id=RUN_ID):
    mlflow.log_metric("overfit_logloss_gap", overfit_gap)
    with tempfile.TemporaryDirectory() as tmp:
        path = os.path.join(tmp, "learning_curves.png")
        fig.savefig(path)
        mlflow.log_artifact(path, "plots")

print(f"Val-Train logloss gap at best iter: {overfit_gap:.6f}")
plt.show()


## 16 — Analysis: Feature Importance (Gain / Weight / Cover)

Pipeline features are shown in blue; enabled NN providers use separate colors.


In [ ]:
importance_types = ["gain", "weight", "cover"]
imp_frames = {}

for imp_type in importance_types:
    scores = model.get_booster().get_score(importance_type=imp_type)
    df_imp = pd.Series(scores, name=imp_type).reindex(feature_cols_ext).fillna(0)
    df_imp = df_imp.sort_values(ascending=False)
    imp_frames[imp_type] = df_imp

imp_combined = pd.DataFrame(imp_frames)
imp_combined.index.name = "feature"

provider_colors = {
    "pipeline": "steelblue",
    "chronos":  "darkorange",
    "kronos":   "seagreen",
    "timesfm":  "mediumpurple",
}

def _feature_color(feature: str) -> str:
    return provider_colors.get(nn_feature_provider_by_col.get(feature, "pipeline"), "gray")

fig, axes = plt.subplots(1, 3, figsize=(16, 7))
for ax, imp_type in zip(axes, importance_types):
    top20 = imp_frames[imp_type].head(20)
    top20.plot.barh(ax=ax, color=[_feature_color(f) for f in top20.index])
    ax.invert_yaxis()
    ax.set_title(f"Importance: {imp_type}")
    ax.set_xlabel("Score")

from matplotlib.patches import Patch
legend_els = [Patch(facecolor=provider_colors["pipeline"], label="Pipeline features")]
legend_els += [
    Patch(facecolor=provider_colors[source], label=f"{source} features")
    for source in nn_feature_frames.keys()
]
axes[1].legend(handles=legend_els, loc="lower right", fontsize=9)

plt.suptitle(
    f"Feature Importance — {DATA_CFG['pair']} {DATA_CFG['timeframe']} + {ENABLED_NN_LABEL}",
    y=1.01,
)
plt.tight_layout()

with mlflow.start_run(run_id=RUN_ID):
    with tempfile.TemporaryDirectory() as tmp:
        img_path = os.path.join(tmp, "feature_importance.png")
        csv_path = os.path.join(tmp, "feature_importance.csv")
        fig.savefig(img_path)
        imp_combined.to_csv(csv_path)
        mlflow.log_artifact(img_path, "plots")
        mlflow.log_artifact(csv_path, "reports")

plt.show()
print("\nTop-10 features by gain:")
imp_combined["gain"].sort_values(ascending=False).head(10)


## 17 — Analysis: Metrics Comparison (Train / Val / Test)


In [ ]:
metrics_df = pd.DataFrame(split_metrics).T
metrics_df.index.name = "split"

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
plot_metrics = [
    (["auc", "avg_precision", "balanced_acc"], "Discrimination"),
    (["f1", "precision", "recall", "mcc"],   "Classification"),
    (["logloss", "brier"],                    "Calibration (lower = better)"),
]

for ax, (cols, title) in zip(axes, plot_metrics):
    metrics_df[cols].plot.bar(ax=ax, rot=0)
    ax.set_title(title)
    ax.set_ylim(0)
    ax.legend(fontsize=8)

plt.suptitle(f"Train / Val / Test Metric Comparison — + {ENABLED_NN_LABEL}", y=1.02)
plt.tight_layout()

with mlflow.start_run(run_id=RUN_ID):
    with tempfile.TemporaryDirectory() as tmp:
        path = os.path.join(tmp, "metrics_comparison.png")
        fig.savefig(path)
        mlflow.log_artifact(path, "plots")

plt.show()
print(metrics_df.round(4).to_string())


## 18 — Run Summary


In [ ]:
required_summary_vars = [
    "RUN_ID", "split_metrics", "EXPERIMENT_NAME", "MLFLOW_DB", "DATA_CFG", "DIAGNOSTIC_CFG", "pd",
    "nn_feature_paths", "feature_cols", "nn_feature_cols", "nn_feature_cols_by_source",
    "feature_cols_ext",
]
missing_summary_vars = [name for name in required_summary_vars if name not in globals()]

if missing_summary_vars:
    print("Run summary is not available yet.")
    print("Missing variables: " + ", ".join(missing_summary_vars))
    print("Run the config, NN feature join, and training/MLflow cells first, then rerun the analysis cells.")
else:
    print("=" * 60)
    print(f"  MLflow Run ID  : {RUN_ID}")
    print(f"  Experiment     : {EXPERIMENT_NAME}")
    print(f"  Tracking DB    : {MLFLOW_DB}")
    print(f"  Primary run    : {DATA_CFG['norm_method']} (d={DATA_CFG.get('fracdiff_d')})")
    print("  Label mode     : binary short/long, neutral dropped")
    print("=" * 60)
    print("  To browse the UI:")
    print(f"  mlflow ui --backend-store-uri {MLFLOW_DB}")
    print("  then open  http://127.0.0.1:5000")
    print("=" * 60)
    print()
    print("NN feature parquets:")
    for source, meta in nn_feature_paths.items():
        print(f"  {source:8s}: {meta['path']}")
    print(f"Pipeline features: {len(feature_cols)}")
    print(f"NN features      : {len(nn_feature_cols)}")
    for source, cols in nn_feature_cols_by_source.items():
        print(f"  {source:8s}: {len(cols)}  {cols}")
    print(f"Total features   : {len(feature_cols_ext)}")
    print()
    print("Logged artifacts:")
    for artifact in [
        "plots/calibration_curve.png",
        "plots/threshold_sweep.png",
        "plots/confidence_band_counts.png",
        "plots/confidence_band_quality.png",
        "plots/confidence_band_calibration.png",
        "plots/confidence_band_confusion_matrices.png",
        "plots/roc_curve.png",
        "plots/pr_curve.png",
        "plots/confusion_matrices.png",
        "plots/binary_prf1.png",
        "plots/learning_curves.png",
        "plots/feature_importance.png",
        "plots/metrics_comparison.png",
        "reports/probability_distribution.csv",
        "reports/threshold_sweep.csv",
        "reports/confidence_band_metrics.csv",
        "reports/classification_report.csv",
        "reports/feature_importance.csv",
        "features.json",
        "model/  (XGBoost model)",
    ]:
        print(f"  {artifact}")

    print()
    test_metrics = split_metrics.get("test", {})
    if test_metrics:
        print("Test metrics:")
        print(pd.Series(test_metrics).round(4).to_string())
    else:
        print("No test metrics were found in split_metrics.")
